# MLFlow Tracking Server on Chameleon

This notebook sets up an MLFlow tracking server on Chameleon with:

- A VM instance at KVM@TACC
- Persistent block storage for PostgreSQL data
- Artifact storage using CHI@TACC S3 object storage

After running this notebook, you will have an MLFlow tracking server accessible via browser.

## Configuration

Set your project ID prefix. All resources (lease, server, volume, bucket) will be named with this prefix. Also set the lease time, i.e. how long do you want your MLFlow service to stay up? (Even after it ends, the data will persist to the next run. You should set the lease duration to the anticipated duration of the training experiments you plan to run, plus some headroom.)

In [ ]:
# Set your project ID prefix (e.g., proj01, proj02, etc.)
# All resources will be named with this prefix
PROJECT_ID = "projXX"  # <-- Change this to your assigned project ID
LEASE_HOURS = 12 # Lease duration in hours (adjust as needed)

In [ ]:
from chi import server, context, lease, network
import chi
import os
import datetime

## Create S3 Bucket and Credentials at CHI@TACC

MLFlow will store artifacts in CHI@TACC S3 object storage. We first create the bucket and credentials.

In [ ]:
context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

In [ ]:
BUCKET_NAME = f"{PROJECT_ID}-mlflow-artifacts"

Create the S3 bucket if it doesn't already exist:

In [ ]:
import swiftclient

os_conn = chi.clients.connection()
token = os_conn.authorize()
storage_url = os_conn.object_store.get_endpoint()

swift_conn = swiftclient.Connection(preauthurl=storage_url,
                                    preauthtoken=token,
                                    retries=5)

# Check if bucket exists, create if not
try:
    swift_conn.head_container(BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except swiftclient.exceptions.ClientException:
    swift_conn.put_container(BUCKET_NAME)
    print(f"Created bucket {BUCKET_NAME}")

Generate EC2 credentials for S3 access. These will be saved to a file so they can be reused.


In [ ]:
ENV_FILE = os.path.expanduser("~/work/.mlflow_s3_credentials")

if os.path.exists(ENV_FILE):
    # Load existing credentials
    credentials = {}
    with open(ENV_FILE, "r") as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                key, value = line.split("=", 1)
                credentials[key] = value
    AWS_ACCESS_KEY_ID = credentials["AWS_ACCESS_KEY_ID"]
    AWS_SECRET_ACCESS_KEY = credentials["AWS_SECRET_ACCESS_KEY"]
    print(f"Loaded existing credentials from {ENV_FILE}")
else:
    # Generate new credentials
    conn = chi.clients.connection()
    project_id = conn.current_project_id
    identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
    url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"
    
    resp = conn.session.post(url, json={"tenant_id": project_id})
    resp.raise_for_status()
    ec2 = resp.json()["credential"]
    
    AWS_ACCESS_KEY_ID = ec2["access"]
    AWS_SECRET_ACCESS_KEY = ec2["secret"]
    
    # Save credentials to file
    os.makedirs(os.path.dirname(ENV_FILE), exist_ok=True)
    with open(ENV_FILE, "w") as f:
        f.write(f"AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}\n")
        f.write(f"AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}\n")
    print(f"Generated new credentials and saved to {ENV_FILE}")

print(f"Bucket: {BUCKET_NAME}")

## Create Lease and Server at KVM@TACC

In [ ]:
context.choose_site(default="KVM@TACC")

Create a 12-hour lease for an m1.medium VM:

In [ ]:
l = lease.Lease(f"{PROJECT_ID}-mlflow-lease", duration=datetime.timedelta(hours=LEASE_HOURS))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.medium"), amount=1)
l.submit(idempotent=True)

In [ ]:
l.show()

Launch the VM instance:

In [ ]:
s = server.Server(
    f"{PROJECT_ID}-mlflow-server",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name
)
s.submit(idempotent=True)

Associate a floating IP:

In [ ]:
s.associate_floating_ip()

Set up security groups to allow SSH, MLFlow UI (port 8000):

In [ ]:
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8000", "port": 8000, "description": "Enable TCP port 8000 (MLFlow UI)"}
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")

## Set up Persistent Volume

We will use a persistent block storage volume for PostgreSQL data. If the volume does not exist, we create it.

In [ ]:
VOLUME_NAME = f"{PROJECT_ID}-mlflow-persist"
VOLUME_SIZE_GB = 5

In [ ]:
cinder_client = chi.clients.cinder()

# Check if volume already exists
existing_volumes = [v for v in cinder_client.volumes.list() if v.name == VOLUME_NAME]

if existing_volumes:
    volume = existing_volumes[0]
    print(f"Found existing volume: {volume.name} (status: {volume.status})")
else:
    volume = cinder_client.volumes.create(name=VOLUME_NAME, size=VOLUME_SIZE_GB)
    print(f"Created new volume: {volume.name}")

In [ ]:
# Wait for volume to be available if just created
import time
volume = cinder_client.volumes.get(volume.id)
while volume.status not in ["available", "in-use"]:
    print(f"Volume status: {volume.status}, waiting...")
    time.sleep(5)
    volume = cinder_client.volumes.get(volume.id)
print(f"Volume status: {volume.status}")

Attach the volume to our server (if not already attached):

In [ ]:
volume = cinder_client.volumes.get(volume.id)
if volume.status == "available":
    s.attach_volume(volume.id)
    print(f"Attached volume {VOLUME_NAME} to server")
else:
    print(f"Volume status is {volume.status}, skipping attach")

## Set up Docker on the Server

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## Mount the Persistent Volume

Format and mount the volume. If the volume already has data, we skip formatting.

In [ ]:
# Check if volume needs formatting (new volume)
# Get the real device path for this attached volume
volume = cinder_client.volumes.get(volume.id)
device = volume.attachments[0]["device"]   # e.g. /dev/vdb
part = f"{device}1"
s.execute("lsblk")

In [ ]:
# Check if partition exists - if not, create it and format
# This will only format if the volume is new (no partition table)
s.execute(f"""
set -e
if ! sudo blkid {part} >/dev/null 2>&1; then
    echo "No filesystem found on {part}; creating partition + ext4"
    sudo parted -s {device} mklabel gpt
    sudo parted -s {device} mkpart primary ext4 0% 100%
    sudo mkfs.ext4 {part}
else
    echo "Filesystem already exists on {part}; skipping format"
fi
sudo mkdir -p /mnt/mlflow_persist
if ! mountpoint -q /mnt/mlflow_persist; then
    sudo mount {part} /mnt/mlflow_persist
fi
sudo chown -R cc:cc /mnt/mlflow_persist
""")

In [ ]:
# Mount the volume
s.execute("""
sudo mkdir -p /mnt/mlflow_persist
sudo mount /dev/vdb1 /mnt/mlflow_persist
sudo chown -R cc:cc /mnt/mlflow_persist
""")

In [ ]:
# Create directory for postgres data
s.execute("mkdir -p /mnt/mlflow_persist/postgres_data")

## Transfer Docker Files to Server

In [ ]:
# Create docker directory on server
s.execute("mkdir -p ~/docker")

In [ ]:
# Create .env file with S3 credentials
env_content = f"""AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}
AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}
BUCKET_NAME={BUCKET_NAME}
"""

with open("docker/.env", "w") as f:
    f.write(env_content)
print("Created docker/.env")

In [ ]:
# Upload docker-compose and .env to server
s.upload("docker/docker-compose-mlflow.yaml", "/home/cc/docker/docker-compose-mlflow.yaml")
s.upload("docker/.env", "/home/cc/docker/.env")
print("Uploaded docker files to server")

In [ ]:
# Verify the files were uploaded
s.execute("ls -la ~/docker/")

## Start MLFlow

Start the MLFlow tracking server with Docker Compose:

In [ ]:
s.execute("docker compose -f ~/docker/docker-compose-mlflow.yaml up -d")

In [ ]:
# Check that containers are running
s.execute("docker ps")

## Access MLFlow UI

Get the floating IP and access MLFlow:

In [ ]:
s.refresh()
floating_ip = s.get_floating_ip()
print(f"MLFlow UI: http://{floating_ip}:8000")
print(f"\nSSH access: ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print(f"\nTo log experiments to this MLFlow server from another machine, set:")
print(f"  export MLFLOW_TRACKING_URI=http://{floating_ip}:8000")

## Cleanup (Optional)

When you are done, you can stop the containers and optionally delete resources.

In [ ]:
# Stop containers
# s.execute("docker compose -f ~/docker/docker-compose-mlflow.yaml down")

In [ ]:
# Unmount volume before detaching
# s.execute("sudo umount /mnt/mlflow_persist")

In [ ]:
# Detach volume (preserves data for next time)
# volume = cinder_client.volumes.get(volume.id)
# s.detach_volume(volume.id)

In [ ]:
# Delete server (preserves volume)
# s.delete()

In [ ]:
# Delete lease
# l.delete()